# L21 — The (s, S) Inventory Control Model

**Module**: M06 | **Chapter**: 8 | **Lecture**: L21

## Learning Objectives
By the end of this notebook you will be able to:
1. Formulate the (s, S) inventory control problem as a DES.
2. Implement the model using `simpy.Container`.
3. Compute holding, ordering, and shortage costs from simulation output.
4. Compare multiple (s, S) policies using 30 replications and confidence intervals.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**
---

In [ ]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from scipy.stats import t as t_dist

## 1. Conceptual Model

**System**: A retailer manages inventory of a single product.

| Component | Description |
|---|---|
| State | Inventory level I(t) ∈ [0, S] |
| Events | Demand arrival (↓ I), Order receipt (↑ I) |
| Policy | If I ≤ s and no pending order: place order for S−I units |
| Costs | h per unit-day (holding), K per order (ordering), p per unit-day short |

**Why SimPy Container?** `simpy.Container` models a bulk resource with a real-valued level, `put()` and `get()` operations. Perfect for inventory.

## 2. Manual Trace (Think → Trace)

Before coding, trace the system by hand.

**Parameters**: s=10, S=50; demand arrives every 3 days, size=10; lead time=5 days

| Time | I before | Event | I after | Order? |
|------|----------|-------|---------|--------|
| 0    | 50       | Start | 50      | No     |
| 3    | 50       | Demand 10 | 40   | No     |
| 6    | 40       | Demand 10 | 30   | No     |
| 9    | 30       | Demand 10 | 20   | No     |
| 12   | 20       | Demand 10 | 10   | Yes (order 40 units) |
| 15   | 10       | Demand 10 | 0    | No (order pending) |
| 17   | 0        | Order arrives | 40 | No  |

**Questions to verify against simulation:**
1. First stockout at what time?
2. Total holding cost over [0, 17]?
3. How many orders placed?

## 3. SimPy Implementation

In [ ]:
@dataclass
class InventoryParams:
    s: float              # reorder point
    S: float              # order-up-to level
    demand_rate: float    # mean demands per time unit
    demand_mean: float    # mean demand quantity per event
    lead_time_mean: float # mean lead time
    holding_cost: float   # per unit per time unit
    order_cost: float     # fixed cost per order
    shortage_cost: float  # per unit-time short
    sim_time: float

def run_inventory(params: InventoryParams, seed: int = 0,
                  track_level: bool = False):
    """Simulate (s,S) inventory; return cost summary and optional level trace."""
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    inv = simpy.Container(env, capacity=params.S, init=params.S)

    # Shared mutable state
    costs = {'holding': 0.0, 'ordering': 0.0, 'shortage': 0.0, 'n_orders': 0}
    pending_order = [False]
    level_trace = []  # (time, level) snapshots

    def demand_process():
        while True:
            yield env.timeout(rng.exponential(1.0 / params.demand_rate))
            demand = max(0.0, rng.exponential(params.demand_mean))
            shortage = max(0.0, demand - inv.level)
            costs['shortage'] += shortage  # instantaneous (simplification)
            actual = min(demand, inv.level)
            if actual > 0:
                yield inv.get(actual)
            # Reorder check
            if inv.level <= params.s and not pending_order[0]:
                pending_order[0] = True
                env.process(order_process())

    def order_process():
        order_qty = params.S - inv.level
        costs['ordering'] += params.order_cost
        costs['n_orders'] += 1
        lead = rng.exponential(params.lead_time_mean)
        yield env.timeout(lead)
        space = inv.capacity - inv.level
        amount = min(order_qty, space)
        if amount > 0:
            yield inv.put(amount)
        pending_order[0] = False

    def holding_accumulator(dt: float = 0.1):
        while True:
            costs['holding'] += inv.level * params.holding_cost * dt
            if track_level:
                level_trace.append((env.now, inv.level))
            yield env.timeout(dt)

    env.process(demand_process())
    env.process(holding_accumulator())
    env.run(until=params.sim_time)

    total = costs['holding'] + costs['ordering'] + costs['shortage']
    result = {
        'total_cost': total,
        'avg_daily_cost': total / params.sim_time,
        'avg_holding': costs['holding'] / params.sim_time,
        'n_orders': costs['n_orders'],
    }
    return (result, level_trace) if track_level else result

In [ ]:
# Quick single run
base_params = InventoryParams(
    s=10, S=50,
    demand_rate=3.0,    # demands/day
    demand_mean=3.0,    # units/demand
    lead_time_mean=5.0, # days
    holding_cost=0.5,   # $/unit/day
    order_cost=100.0,   # $/order
    shortage_cost=10.0, # $/unit short
    sim_time=365.0,
)

result, trace = run_inventory(base_params, seed=42, track_level=True)
print("Single-run result (s=10, S=50):")
for k, v in result.items():
    print(f"  {k:20s}: {v:.2f}" if isinstance(v, float) else f"  {k:20s}: {v}")

## 4. Inventory Level Trace — The Sawtooth Pattern

In [ ]:
trace_df = pd.DataFrame(trace, columns=['time', 'level'])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(trace_df['time'], trace_df['level'], color='steelblue', lw=0.8)
ax.axhline(base_params.s, color='red', linestyle='--',
           linewidth=1.2, label=f's={base_params.s} (reorder point)')
ax.axhline(base_params.S, color='green', linestyle='--',
           linewidth=1.2, label=f'S={base_params.S} (order-up-to)')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Inventory level (units)')
ax.set_title(f'(s={base_params.s}, S={base_params.S}) Inventory — Sawtooth Pattern')
ax.legend()
# Show only first 120 days
ax.set_xlim(0, 120)
plt.tight_layout()
plt.show()

## 5. Scenario Comparison: Three (s, S) Policies

In [ ]:
from dataclasses import replace

policies = [
    (5,  40, 'Aggressive (low s, S)'),
    (10, 50, 'Base (s=10, S=50)'),
    (20, 60, 'Conservative (high s, S)'),
]

N_REPS = 30
all_results = []

for s, S, label in policies:
    params = replace(base_params, s=s, S=S, sim_time=365.0)
    rep_costs = []
    for rep in range(N_REPS):
        r = run_inventory(params, seed=rep)
        rep_costs.append(r['avg_daily_cost'])
    costs_arr = np.array(rep_costs)
    m = costs_arr.mean()
    se = costs_arr.std() / np.sqrt(N_REPS)
    hw = t_dist.ppf(0.975, df=N_REPS-1) * se
    all_results.append({
        'policy': label,
        's': s, 'S': S,
        'mean_cost': m,
        'ci_lower': m - hw,
        'ci_upper': m + hw,
    })
    print(f"{label}: ${m:.2f}/day  95% CI=[{m-hw:.2f}, {m+hw:.2f}]")

results_df = pd.DataFrame(all_results)

In [ ]:
# Bar chart with CI error bars
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for i, row in results_df.iterrows():
    ax.bar(i, row['mean_cost'], color=colors[i], alpha=0.7,
           label=row['policy'])
    ax.errorbar(i, row['mean_cost'],
                yerr=[[row['mean_cost'] - row['ci_lower']],
                      [row['ci_upper'] - row['mean_cost']]],
                fmt='none', color='black', capsize=5, lw=1.5)

ax.set_xticks(range(len(results_df)))
ax.set_xticklabels([f"(s={r['s']}, S={r['S']})" for _, r in results_df.iterrows()])
ax.set_ylabel('Average daily cost ($/day)')
ax.set_title('(s, S) Policy Comparison — 30 Replications (T=365 days)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Using the simdes Package

In [ ]:
from simdes.models.inventory import SSInventory

model = SSInventory(
    reorder_point=10, order_up_to=50,
    demand_rate=3.0,
    sim_time=365.0,
    seed=0,
)
rep_df = model.run_replications(n=30)
print(rep_df.describe().round(3))

---
## Try It Yourself

1. **Cost breakdown**: Modify `run_inventory` to also return average holding cost, average ordering cost, and average shortage cost per day. How does the cost composition differ across the three policies?

2. **Lead time sensitivity**: Fix (s=10, S=50) and vary `lead_time_mean` ∈ {1, 3, 5, 10}. Plot average daily cost vs lead time. At what lead time does shortage cost become the dominant component?

3. **Grid search**: Simulate all (s, S) combinations where s ∈ {5, 10, 15, 20} and S ∈ {40, 50, 60} (12 combinations). Use 30 replications each. Plot a heatmap of average daily cost. Which policy minimises cost?

4. **Bridge to RL**: In Chapter 14, a reinforcement learning agent will learn the (s, S) parameters automatically. Describe in one paragraph what information the agent would receive as its *state*, what *actions* it could take, and what *reward* would encode minimising cost.